# HW1: K-Anonymity Experiment
**Dataset**: Adult Census Income  
**Method**: Mondrian Multidimensional K-Anonymity  
**Goal**: Observe how K-anonymity impacts ML performance

## Step 1: Data Preprocessing

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('adult.csv')
print('Original shape:', df.shape)

# Remove rows with '?' values
df = df.replace('?', np.nan).dropna().reset_index(drop=True)
print('After removing missing values:', df.shape)

# Encode target variable: income -> 0/1
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})
print('Income distribution:')
print(df['income'].value_counts())

# Column roles
NUMERIC_QI          = ['age', 'educational-num', 'hours-per-week']
CATEGORICAL_QI      = ['gender', 'race', 'marital-status']
QI_COLUMNS          = NUMERIC_QI + CATEGORICAL_QI
TARGET              = 'income'
DROP_COLUMNS_FOR_ML = ['education']          # redundant with educational-num
SUPPRESS_COLUMNS    = ['native-country']     # kept in data, but all values -> '*'

print('\nQI columns      :', QI_COLUMNS)
print('Drop for ML     :', DROP_COLUMNS_FOR_ML)
print('Suppress to *   :', SUPPRESS_COLUMNS)

df.to_csv('adult_cleaned.csv', index=False)
print('\nSaved adult_cleaned.csv')
df.head()

Original shape: (48842, 15)
After removing missing values: (45222, 15)
Income distribution:
income
0    34014
1    11208
Name: count, dtype: int64

QI columns      : ['age', 'educational-num', 'hours-per-week', 'gender', 'race', 'marital-status']
Drop for ML     : ['education']
Suppress to *   : ['native-country']

Saved adult_cleaned.csv


,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,0


## Step 2: Define Generalization Hierarchies
Only categorical QI columns need a hierarchy.  
Each hierarchy is a **list of dicts**, from fine to coarse (each dict maps `value -> parent`).

In [2]:
HIERARCHIES = {

    # gender: 2 values -> *
    'gender': [
        {'Male': '*', 'Female': '*'}
    ],

    # race: 5 values -> 2 groups -> *
    'race': [
        {
            'White':               'White',
            'Black':               'Non-White',
            'Asian-Pac-Islander':  'Non-White',
            'Amer-Indian-Eskimo':  'Non-White',
            'Other':               'Non-White',
        },
        {'White': '*', 'Non-White': '*'}
    ],

    # marital-status: 7 values -> 3 groups -> *
    'marital-status': [
        {
            'Married-civ-spouse':    'Married',
            'Married-AF-spouse':     'Married',
            'Never-married':         'Single',
            'Widowed':               'Single',
            'Divorced':              'Separated-Divorced',
            'Separated':             'Separated-Divorced',
            'Married-spouse-absent': 'Separated-Divorced',
        },
        {'Married': '*', 'Single': '*', 'Separated-Divorced': '*'}
    ],
}

# Sanity check: every categorical value in the data is covered
for col in CATEGORICAL_QI:
    if col in HIERARCHIES:
        covered = set(HIERARCHIES[col][0].keys())
        actual  = set(df[col].unique())
        missing = actual - covered
        print(f'{col:20s}: {"OK" if not missing else "MISSING: " + str(missing)}')

gender              : OK
race                : OK
marital-status      : OK


## Step 3: Mondrian Multidimensional K-Anonymity

In [3]:
class MondrianAnonymizer:
    """
    Mondrian Multidimensional K-Anonymity
    Reference: LeFevre et al., ICDE 2006

    Numeric QI     -> generalized as range strings, e.g. '25-40'
    Categorical QI -> generalized via user-defined hierarchy
    """

    def __init__(self, k, numeric_qi, categorical_qi, hierarchies=None):
        self.k              = k
        self.numeric_qi     = numeric_qi
        self.categorical_qi = categorical_qi
        self.qi_columns     = numeric_qi + categorical_qi
        self.hierarchies    = hierarchies or {}

    # ------------------------------------------------------------------ #
    def fit_transform(self, df):
        """Return a new dataframe with QI columns anonymized."""
        self._global = {}
        for col in self.numeric_qi:
            lo, hi = df[col].min(), df[col].max()
            self._global[col] = hi - lo if hi != lo else 1
        for col in self.categorical_qi:
            self._global[col] = df[col].nunique() or 1

        self._orig   = df
        self._result = df.copy()
        # Cast QI columns to object so range strings can be assigned
        for col in self.qi_columns:
            self._result[col] = self._result[col].astype(object)

        self._anonymize(list(df.index))
        return self._result

    # ------------------------------------------------------------------ #
    def _anonymize(self, indices):
        if len(indices) < self.k:
            self._generalize(indices)
            return
        dims = sorted(self.qi_columns,
                      key=lambda c: self._span(indices, c), reverse=True)
        for col in dims:
            split = self._find_split(indices, col)
            if split:
                self._anonymize(split[0])
                self._anonymize(split[1])
                return
        self._generalize(indices)

    def _span(self, indices, col):
        vals = self._orig.loc[indices, col]
        if col in self.numeric_qi:
            return (vals.max() - vals.min()) / self._global[col]
        return vals.nunique() / self._global[col]

    def _find_split(self, indices, col):
        vals = self._orig.loc[indices, col]
        if col in self.numeric_qi:
            median = vals.median()
            lhs = [i for i in indices if self._orig.at[i, col] <= median]
            rhs = [i for i in indices if self._orig.at[i, col] >  median]
        else:
            unique_sorted = sorted(vals.unique())
            if len(unique_sorted) < 2:
                return None
            mid     = len(unique_sorted) // 2
            lhs_set = set(unique_sorted[:mid])
            rhs_set = set(unique_sorted[mid:])
            lhs = [i for i in indices if self._orig.at[i, col] in lhs_set]
            rhs = [i for i in indices if self._orig.at[i, col] in rhs_set]
        return (lhs, rhs) if len(lhs) >= self.k and len(rhs) >= self.k else None

    def _generalize(self, indices):
        for col in self.qi_columns:
            vals = self._orig.loc[indices, col]
            if col in self.numeric_qi:
                lo, hi = int(vals.min()), int(vals.max())
                summary = f'{lo}-{hi}' if lo != hi else str(lo)
            else:
                unique_vals = vals.unique().tolist()
                summary = unique_vals[0] if len(unique_vals) == 1 \
                          else self._lca(col, unique_vals)
            self._result.loc[indices, col] = summary

    def _lca(self, col, values):
        if col not in self.hierarchies:
            return '*'
        cur = {v: v for v in values}
        for level_map in self.hierarchies[col]:
            cur = {v: level_map.get(cur[v], '*') for v in values}
            if len(set(cur.values())) == 1:
                return list(cur.values())[0]
        return '*'

print('MondrianAnonymizer defined.')

MondrianAnonymizer defined.


## Step 4: Apply K-Anonymity for Multiple K Values

In [4]:
import time

K_VALUES = [2, 5, 10, 25, 50]
anonymized_dfs = {}

df_clean = pd.read_csv('adult_cleaned.csv')

print(f"{'k':>4} | {'groups':>7} | {'min_size':>8} | {'time':>6} | status")
print('-' * 45)

for k in K_VALUES:
    t0 = time.time()
    df_anon = MondrianAnonymizer(
        k=k,
        numeric_qi=NUMERIC_QI,
        categorical_qi=CATEGORICAL_QI,
        hierarchies=HIERARCHIES
    ).fit_transform(df_clean)

    # Full suppression: set suppress columns to '*'
    for col in SUPPRESS_COLUMNS:
        df_anon[col] = '*'

    anonymized_dfs[k] = df_anon
    df_anon.to_csv(f'adult_k{k}.csv', index=False)

    group_sizes = df_anon.groupby(QI_COLUMNS).size()
    min_size    = group_sizes.min()
    status      = 'PASS' if min_size >= k else 'FAIL'
    print(f'{k:4d} | {len(group_sizes):7d} | {min_size:8d} | {time.time()-t0:5.1f}s | [{status}]')

print('\nDone! CSV files saved.')

   k |  groups | min_size |   time | status
---------------------------------------------
   2 |    6667 |        2 |  54.0s | [PASS]
   5 |    3401 |        5 |  28.4s | [PASS]
  10 |    2031 |       10 |  19.8s | [PASS]
  25 |     966 |       25 |  13.6s | [PASS]
  50 |     554 |       50 |  10.3s | [PASS]

Done! CSV files saved.


In [5]:
# Inspect anonymized output (k=10)
k_ex = 10
df_ex = anonymized_dfs[k_ex]
print(f'Sample rows (k={k_ex}):')
display(df_ex[QI_COLUMNS + [TARGET]].head(10))
print()
for col in NUMERIC_QI:
    print(f'{col} unique (first 10):', sorted(df_ex[col].unique())[:10])
print()
for col in CATEGORICAL_QI:
    print(f'{col} unique:', sorted(df_ex[col].unique()))

Sample rows (k=10):


,age,educational-num,hours-per-week,gender,race,marital-status,income
0,25,5-9,23-40,Male,Black,Never-married,0
1,38,7-9,50,Male,White,Married-civ-spouse,0
2,28-29,11-13,20-40,Male,*,Married-civ-spouse,1
3,44-46,10,20-40,Male,Black,*,1
4,34,3-9,25-40,Male,White,Never-married,0
5,63-66,14-16,32-40,Male,*,Married-civ-spouse,1
6,24,10,35-40,Female,White,Never-married,0
7,55-56,2-9,8-40,Male,White,Married-civ-spouse,0
8,65,2-9,32-40,Male,White,Married-civ-spouse,1
9,36-37,11-13,5-40,Male,*,*,0



age unique (first 10): ['17', '17-18', '17-19', '17-20', '17-21', '17-22', '17-23', '17-24', '17-26', '17-34']
educational-num unique (first 10): ['1-10', '1-4', '1-5', '1-6', '1-8', '1-9', '10', '11', '11-12', '11-13']
hours-per-week unique (first 10): ['1-18', '1-20', '1-36', '1-37', '1-40', '10', '10-20', '10-21', '10-22', '10-25']

gender unique: ['Female', 'Male']
race unique: ['*', 'Amer-Indian-Eskimo', 'Asian-Pac-Islander', 'Black', 'Non-White', 'Other', 'White']
marital-status unique: ['*', 'Divorced', 'Married', 'Married-civ-spouse', 'Married-spouse-absent', 'Never-married', 'Separated', 'Single', 'Widowed']
